In [1]:
import os
import re
from pathlib import Path
import mne
import numpy as np
from mne.preprocessing import ICA
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

In [7]:
DATA_DIR = Path(r"C:\EEG Classification\data\raw")  # Adjust to your folder path

SCALP_CHANNELS = [
    "EEG Fp1-LE", "EEG F3-LE", "EEG C3-LE", "EEG P3-LE", "EEG O1-LE",
    "EEG F7-LE",  "EEG T3-LE", "EEG T5-LE", "EEG Fz-LE", "EEG Fp2-LE",
    "EEG F4-LE",  "EEG C4-LE", "EEG P4-LE", "EEG O2-LE", "EEG F8-LE",
    "EEG T4-LE",  "EEG T6-LE", "EEG Cz-LE", "EEG Pz-LE"
]

In [2]:
def preprocess(file_path, scalp_channels, l_freq=1.0, h_freq=45.0, notch_freq=50.0):

    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    
    rename_dict = {ch: ch.replace("EEG ", "").replace("-LE", "").strip() for ch in raw.ch_names}
    raw.rename_channels(rename_dict)
    clean_scalp_ch = [ch.replace("EEG ", "").replace("-LE", "").strip() for ch in scalp_channels]
    raw.pick(clean_scalp_ch)
    
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)
    raw.notch_filter(freqs=notch_freq, verbose=False)
    
    data = raw.get_data()
    ch_ptps = np.ptp(data, axis=1)
    median_ptp = np.median(ch_ptps)
    std_ptp = np.std(ch_ptps)
    
    bad_channels = [
        raw.ch_names[i] for i, ptp in enumerate(ch_ptps) 
        if ptp > (median_ptp + 5 * std_ptp) or ptp < 1e-6
    ]
    raw.info['bads'] = bad_channels

    # Independent Component Analysis (ICA)
    n_components = min(15, len(raw.ch_names) - len(bad_channels))
    ica = ICA(n_components=n_components, random_state=42, max_iter=800)
    ica.fit(raw, picks='eeg', reject_by_annotation=True, verbose=False)
    
    # Auto-detect ocular blink components using frontal sensors
    eog_indices = []
    for test_ch in ['Fp1', 'Fp2']:
        if test_ch in raw.ch_names and test_ch not in raw.info['bads']:
            scores = ica.score_sources(raw, target=test_ch, score_func='pearsonr', verbose=False)
            top_matches = np.where(np.abs(scores) > 0.40)[0].tolist()
            eog_indices.extend(top_matches)
            
    ica.exclude = list(set(eog_indices))[:2]
    raw_cleaned = raw.copy()
    ica.apply(raw_cleaned, verbose=False)

    if len(raw_cleaned.info['bads']) > 0:
        raw_cleaned.interpolate_bads(reset_bads=True, verbose=False)
    raw_cleaned.set_eeg_reference('average', projection=False, verbose=False)

    epochs = mne.make_fixed_length_epochs(
        raw_cleaned, duration=4.0, overlap=1.0, preload=True, verbose=False
    )
    
    epochs.drop_bad(reject=dict(eeg=250e-6), verbose=False)
    return epochs

In [4]:
def extract_features(epochs):
    spectrum = epochs.compute_psd(method='welch', fmin=1, fmax=45, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)
    
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }
    
    total_mask = (freqs >= 1) & (freqs <= 45)
    total_power = np.trapezoid(psds[:, :, total_mask], freqs[total_mask], axis=2)
    
    features = []
    for _, (low, high) in bands.items():
        b_mask = (freqs >= low) & (freqs < high)
        bp = np.trapezoid(psds[:, :, b_mask], freqs[b_mask], axis=2)
        rel_bp = bp / (total_power + 1e-12)
        features.append(rel_bp)
        
    X_psd = np.concatenate(features, axis=1)  # (n_epochs, n_channels * 5)
    
    # Calculate Frontal Alpha Asymmetry (F4 vs F3)
    ch_names = epochs.ch_names
    if 'F4' in ch_names and 'F3' in ch_names:
        f4_idx, f3_idx = ch_names.index('F4'), ch_names.index('F3')
        a_mask = (freqs >= 8) & (freqs < 13)
        alpha_f4 = np.trapezoid(psds[:, f4_idx, a_mask], freqs[a_mask], axis=1)
        alpha_f3 = np.trapezoid(psds[:, f3_idx, a_mask], freqs[a_mask], axis=1)
        faa = (np.log(alpha_f4 + 1e-12) - np.log(alpha_f3 + 1e-12))[:, None]
        X_psd = np.hstack([X_psd, faa])
        
    return X_psd

In [5]:
def build_dataset(data_dir, scalp_channels):
    
    X_list, y_list, group_list = [], [], []
    files = list(Path(data_dir).glob("*.edf"))
    print(f"Found {len(files)} EDF files in {data_dir}...")
    
    for f in files:
        label = 1 if "MDD" in f.name.upper() else 0
        
        tokens = f.name.split()
        subject_id = f"{tokens[0]}_{tokens[1]}" if len(tokens) >= 2 else f.stem
        
        try:
            epochs = preprocess(f, scalp_channels)
            if len(epochs) < 5:
                print(f"Skipping {f.name}: only {len(epochs)} valid epochs retained.")
                continue
            
            feats = extract_features(epochs)
            X_list.append(feats)
            
            y_list.extend([label] * len(epochs))
            group_list.extend([subject_id] * len(epochs))
            print(f"Processed {f.name} -> {len(epochs)} epochs (Subject: {subject_id}, Label: {label})")
        except Exception as e:
            print(f"Error reading {f.name}: {e}")
            
    X = np.vstack(X_list)
    y = np.array(y_list)
    groups = np.array(group_list)
    
    return X, y, groups

In [6]:
def train(X_ml, y, groups):
    print("RUNNING XGBOOST")
    
    gkf = GroupKFold(n_splits=5)
    accs, f1s, aucs = [], [], []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_ml, y, groups=groups)):
        X_train, X_val = X_ml[train_idx], X_ml[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        scaler = RobustScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        
        clf = XGBClassifier(
            n_estimators=150, max_depth=4, learning_rate=0.05, 
            eval_metric='logloss', random_state=42
        )
        clf.fit(X_train, y_train)
        preds = clf.predict(X_val)
        probs = clf.predict_proba(X_val)[:, 1]
        
        fold_acc = accuracy_score(y_val, preds)
        fold_f1 = f1_score(y_val, preds)
        fold_auc = roc_auc_score(y_val, probs)
        
        accs.append(fold_acc)
        f1s.append(fold_f1)
        aucs.append(fold_auc)
        print(f"Fold {fold + 1} -> Accuracy: {fold_acc*100:.2f}%, F1: {fold_f1:.3f}, AUC: {fold_auc:.3f}")

    mean_acc = np.mean(accs)*100
    mean_std = np.std(accs)*100
    mean_f1 = np.mean(f1s)
    mean_auc = np.mean(aucs)
    print(f"\n--- XGBoost 5-Fold Summary ---")
    print(f"Mean Accuracy: {mean_acc:.2f}% ± {mean_std:.2f}%")
    print(f"Mean F1-Score: {mean_f1:.3f} | Mean ROC-AUC: {mean_auc:.3f}")


In [8]:
X, y, groups = build_dataset(DATA_DIR, SCALP_CHANNELS)
print(f"\nExtracted Dataset Shapes:")
print(f"ML Matrix (X): {X.shape}")
print(f"Total Epochs: {len(y)} | Unique Subjects: {len(np.unique(groups))}")

train(X, y, groups)

Found 181 EDF files in C:\EEG Classification\data\raw...
Processed 6921959_H S15 EO.edf -> 100 epochs (Subject: 6921959_H_S15, Label: 0)
Processed H S1 EC.edf -> 98 epochs (Subject: H_S1, Label: 0)
Processed H S1 EO.edf -> 114 epochs (Subject: H_S1, Label: 0)
Processed H S1 TASK.edf -> 199 epochs (Subject: H_S1, Label: 0)
Processed H S10 EC.edf -> 125 epochs (Subject: H_S10, Label: 0)
Processed H S10 EO.edf -> 99 epochs (Subject: H_S10, Label: 0)
Processed H S10 TASK.edf -> 199 epochs (Subject: H_S10, Label: 0)
Processed H S11 EC.edf -> 97 epochs (Subject: H_S11, Label: 0)
Processed H S11 EO.edf -> 100 epochs (Subject: H_S11, Label: 0)
Processed H S11 TASK.edf -> 199 epochs (Subject: H_S11, Label: 0)
Processed H S12 EO.edf -> 99 epochs (Subject: H_S12, Label: 0)
Processed H S12 TASK.edf -> 199 epochs (Subject: H_S12, Label: 0)
Processed H S13 EC.edf -> 97 epochs (Subject: H_S13, Label: 0)
Processed H S13 EO.edf -> 100 epochs (Subject: H_S13, Label: 0)
Processed H S13 TASK.edf -> 203 ep

C:\Users\BITPATNA\AppData\Roaming\Python\Python313\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Processed H S14 EC.edf -> 100 epochs (Subject: H_S14, Label: 0)


C:\Users\BITPATNA\AppData\Roaming\Python\Python313\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Processed H S14 EO.edf -> 56 epochs (Subject: H_S14, Label: 0)


C:\Users\BITPATNA\AppData\Roaming\Python\Python313\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


... (7 outputs left out)

C:\Users\BITPATNA\AppData\Roaming\Python\Python313\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Processed H S8 TASK.edf -> 195 epochs (Subject: H_S8, Label: 0)
Processed H S9 EC.edf -> 100 epochs (Subject: H_S9, Label: 0)
Processed H S9 EO.edf -> 100 epochs (Subject: H_S9, Label: 0)
Processed H S9 TASK.edf -> 195 epochs (Subject: H_S9, Label: 0)
Processed MDD S1  EO.edf -> 88 epochs (Subject: MDD_S1, Label: 1)
Processed MDD S1 EC.edf -> 59 epochs (Subject: MDD_S1, Label: 1)
Processed MDD S1 TASK.edf -> 188 epochs (Subject: MDD_S1, Label: 1)
Processed MDD S10 EC.edf -> 100 epochs (Subject: MDD_S10, Label: 1)
Processed MDD S10 EO.edf -> 99 epochs (Subject: MDD_S10, Label: 1)
Processed MDD S10 TASK.edf -> 205 epochs (Subject: MDD_S10, Label: 1)
Processed MDD S11  EC.edf -> 99 epochs (Subject: MDD_S11, Label: 1)
Processed MDD S11 EO.edf -> 99 epochs (Subject: MDD_S11, Label: 1)
Processed MDD S11 TASK.edf -> 208 epochs (Subject: MDD_S11, Label: 1)
Processed MDD S12 EO.edf -> 62 epochs (Subject: MDD_S12, Label: 1)
Processed MDD S12 TASK.edf -> 214 epochs (Subject: MDD_S12, Label: 1)
Pr

C:\Users\BITPATNA\AppData\Roaming\Python\Python313\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Processed MDD S16 TASK.edf -> 211 epochs (Subject: MDD_S16, Label: 1)
Processed MDD S17 EC.edf -> 98 epochs (Subject: MDD_S17, Label: 1)
Processed MDD S17 EO.edf -> 99 epochs (Subject: MDD_S17, Label: 1)
Processed MDD S17 TASK.edf -> 209 epochs (Subject: MDD_S17, Label: 1)
Processed MDD S18 EC.edf -> 99 epochs (Subject: MDD_S18, Label: 1)
Processed MDD S18 EO.edf -> 96 epochs (Subject: MDD_S18, Label: 1)
Processed MDD S18 TASK.edf -> 209 epochs (Subject: MDD_S18, Label: 1)
Processed MDD S19 EC.edf -> 99 epochs (Subject: MDD_S19, Label: 1)
Processed MDD S19 EO.edf -> 98 epochs (Subject: MDD_S19, Label: 1)
Processed MDD S19 TASK.edf -> 211 epochs (Subject: MDD_S19, Label: 1)
Processed MDD S2  EC.edf -> 90 epochs (Subject: MDD_S2, Label: 1)
Processed MDD S2 EO.edf -> 98 epochs (Subject: MDD_S2, Label: 1)
Processed MDD S2 TASK.edf -> 211 epochs (Subject: MDD_S2, Label: 1)
Processed MDD S20 EC.edf -> 87 epochs (Subject: MDD_S20, Label: 1)
Processed MDD S20 EO.edf -> 97 epochs (Subject: MDD_

In [9]:
X.size

2272032

In [10]:
X.shape

(23667, 96)

In [11]:
y.shape

(23667,)